In [ ]:
import json
import re
import requests
import os
from typing import List

from prompts import MAIN_SYSTEM_PROMPT, MAIN_USER_PROMPT


# ==========================================
# 1️⃣  DeepSeek LLM 调用
# ==========================================
def call_llm(
    model: str,
    api_key: str,
    user_prompt: str,
    system_prompt: str,
    base_url: str = "https://api.deepseek.com/v1",
    temperature: float = 0,
    max_tokens: int = 2000,
):
    url = f"{base_url}/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]


# ==========================================
# 2️⃣  文本分块（简单长度分割）
# ==========================================
def chunk_text(text: str, chunk_size: int = 1200) -> List[str]:
    """
    按字符长度分块（极简实现）
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end
    return chunks


# ==========================================
# 3️⃣  从 LLM 输出中提取 JSON
# ==========================================
def extract_json(text: str):
    json_pattern = r"\[.*?\]"
    match = re.search(json_pattern, text, re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group())
    except Exception:
        return None


# ==========================================
# 4️⃣  实体标准化（第二次 LLM 调用）
# ==========================================
STANDARDIZE_SYSTEM_PROMPT = """
你是一个知识图谱实体标准化助手。
任务：
1. 合并同义实体
2. 统一表达形式
3. 不要改变语义
4. 输出仍然为 JSON 数组格式
"""

def standardize_entities(triples, api_key):
    user_prompt = f"""
以下是因果三元组，请对实体进行标准化：

{json.dumps(triples, ensure_ascii=False, indent=2)}

输出格式保持不变。
"""

    raw_output = call_llm(
        model="deepseek-chat",
        api_key=api_key,
        user_prompt=user_prompt,
        system_prompt=STANDARDIZE_SYSTEM_PROMPT,
    )

    normalized = extract_json(raw_output)
    return normalized if normalized else triples


# ==========================================
# 5️⃣  抽取单个 chunk 的因果三元组
# ==========================================
def extract_from_chunk(text: str, api_key: str):
    user_prompt = MAIN_USER_PROMPT + f"```\n{text}\n```"

    raw_output = call_llm(
        model="deepseek-chat",
        api_key=api_key,
        user_prompt=user_prompt,
        system_prompt=MAIN_SYSTEM_PROMPT,
    )

    triples = extract_json(raw_output)
    if not triples:
        return []

    clean_triples = []
    for item in triples:
        if (
            isinstance(item, dict)
            and "cause" in item
            and "relationship_type" in item
            and "effect" in item
        ):
            clean_triples.append(item)

    return clean_triples


# ==========================================
# 6️⃣  主流程
# ==========================================
def extract_triples(text: str, api_key: str):
    print("\n📦 开始分块...")
    chunks = chunk_text(text)

    all_triples = []

    for i, chunk in enumerate(chunks):
        print(f"\n🔹 处理第 {i+1} 块")
        triples = extract_from_chunk(chunk, api_key)
        all_triples.extend(triples)

    print("\n🔧 实体标准化中...")
    all_triples = standardize_entities(all_triples, api_key)

    return all_triples


# ==========================================
# 7️⃣  CLI 入口
# ==========================================
if __name__ == "__main__":
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise ValueError("请设置环境变量 DEEPSEEK_API_KEY")

    input_text = input("请输入文本：\n")

    triples = extract_triples(input_text, api_key)

    print("\n===== 最终因果三元组 =====\n")
    print(json.dumps(triples, indent=2, ensure_ascii=False))